# Correlation of instances with signer counts

| EXP NO    | Model    | Metric     | Split         | 
|-----------|----------|------------|---------------|
| 000       | MViTv2_S | F1-score   | WLASL-2000    |

Independant variables on the `train` set (per class):
- Number of instances 
- Number of unique signers


Using the preprocessed split where videos with <= 9 frames were removed:
|Split      |  Video id             |
|-----------|-----------------------|
| asl1000   | 18223, 59958          |
| asl2000   | 18223, 59958, 15144   |

If frame start or frame end were labeled wrong, they were set to 0 or frame length

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import src.preprocess as preproc
from src.configs import get_class_list
from src.run_types import AVAIL_SETS, CUTOFF_9_NAMES, CUTOFF_SPLITS, RESULTS_OUTPUTS
from src.stats import (
    create_instances_table,
    get_min_max_num_instances,
    get_min_max_num_signers,
)
from src.video_dataset import get_wlasl_info
from src.visualise2 import (
    DEFAULT_ACCENT,
    LINE_PALETTE,
    plot_metric_correlation,
    save_fig,
    set_thesis_style,
)


## Set params

In [ ]:
# split_options: list[AVAIL_SPLITS] = ["asl100", "asl300", "asl1000", "asl2000"]
split_options: list[CUTOFF_SPLITS] = CUTOFF_9_NAMES
set_options: list[AVAIL_SETS] = ['train', 'test', 'val']
split_name : CUTOFF_SPLITS = 'asl2000_cutoff_9'
set_name : AVAIL_SETS = 'train'
metric = 'num_instances'
classes = get_class_list()
set_thesis_style()

project_name = 'satnac_2026'
output_dir = RESULTS_OUTPUTS / project_name

print(f'Split: {split_name} Set: {set_name}')

Split: asl2000_cutoff_9 Set: train


## Load stats

Dataset information:

In [3]:
dataset_info = get_wlasl_info(split_name, set_name)
print(json.dumps({k : str(v) for k, v in dataset_info.items()}, indent=4))

{
    "root": "/home/luke/Code/SLR/data/WLASL/WLASL2000",
    "labels": "/home/luke/Code/SLR/data/WLASL/preprocessed/labels/asl2000_cutoff_9",
    "label_suff": "fixed_frange_bboxes.json",
    "set_name": "train"
}


### Show variation of datasets in a table

In [4]:
from src.stats import (
    WLASLClass,
    get_set_stats,
    retrieve_split_data,
    reverse_preproc_format,
)

all_splits = {split : {} for split in split_options}

set_name : AVAIL_SETS = 'train'
for split_name in split_options:
    # split_dir = labels_dir / split_name
    dataset_info = get_wlasl_info(split_name, set_name)
    split_dir = dataset_info['labels']
    labels_dir = dataset_info['labels'].parent
    label_suffix = dataset_info['label_suff']
    set_info = retrieve_split_data(split=split_name, labels_dir=labels_dir, pattern=f'*{label_suffix}')
    seperated = {}
    set_keys = ['train', 'test', 'val']
    print(f'Split info for {split_name}')
    for  key, value in set_info.items():
        preproc_set = [preproc.Instance.model_validate(v) for v in value]
        
        for set_key in set_keys:
            if key.startswith(set_key):
                seperated[set_key] = preproc_set
                break
    
    sep_wlasl_order = {}
    for key, value in seperated.items():
        sep_wlasl_order[key] = reverse_preproc_format(value, classes=classes)

    
    per_set_stats = {}
    for set_name, glosses_subset in sep_wlasl_order.items():
        per_set_stats[set_name] = get_set_stats([WLASLClass.model_validate(g) for g in glosses_subset])

    all_splits[split_name] = per_set_stats
    
    display(create_instances_table(per_set_stats))
    print('\n', '-'*10)

Split info for asl100_cutoff_9


,Set name,Num instances,Num signers,instances per gloss,signers per gloss
0,train,1442,91,[12 - 30],[7 - 15]
1,test,258,56,[2 - 5],[2 - 4]
2,val,338,69,[3 - 6],[2 - 6]



 ----------
Split info for asl300_cutoff_9


,Set name,Num instances,Num signers,instances per gloss,signers per gloss
0,train,3549,104,[9 - 30],[5 - 15]
1,test,668,74,[2 - 5],[1 - 4]
2,val,901,88,[2 - 6],[2 - 6]



 ----------
Split info for asl1000_cutoff_9


,Set name,Num instances,Num signers,instances per gloss,signers per gloss
0,train,8977,113,[6 - 30],[4 - 15]
1,test,1876,89,[1 - 5],[1 - 4]
2,val,2319,101,[1 - 6],[1 - 6]



 ----------
Split info for asl2000_cutoff_9


,Set name,Num instances,Num signers,instances per gloss,signers per gloss
0,train,14290,117,[1 - 30],[1 - 15]
1,test,2879,105,[1 - 5],[1 - 4]
2,val,3917,109,[1 - 6],[1 - 6]



 ----------


## Plot correlation of F1-score with number of examples, and number of unique signer ids.

In [ ]:
from typing import cast

num_classes = 2000
per_set_stats = all_splits[split_name]
exp_no = '000'
model_name = 'MViTv2_S' # can also use MViTv2_S_e and S3D
checkpoint_num = None # override if multitple runs of the same experiment
split_name : CUTOFF_SPLITS = 'asl2000_cutoff_9'

['000']


### Fetch results

In [ ]:
# write files for easy access when finished
res_out_dir = Path("./outputs/")
res_out_dir.mkdir(parents=True, exist_ok=True)

stub = f"{model_name}_{split_name}_{exp_no}_{str(checkpoint_num) + '_' if checkpoint_num is not None else ''}"
out1 = f"{stub}best_val_loss.json"
out2 = f"{stub}cls_rep_all_targets_preds.json"

#### Run tests if not already done

In [ ]:
def run_tests():
    from src.testing import (
        DATA_FNAME,
        MinInfo,
        get_model_checkpoint_dir,
        get_model_exp_dir,
        load_test_sizes,
        test_run,
    )

    # Load model and data + evaluate
    output = get_model_exp_dir(split=split_name, model=model_name, exp_no=int(exp_no))
    save_path = get_model_checkpoint_dir(output, checkpoint_num)
    admin = MinInfo(model=model_name, split=split_name, save_path=str(save_path))

    data_info_path = output / DATA_FNAME
    data = load_test_sizes(output)
    print(f"Loaded data info from {data_info_path}")

    results, cls_report, all_targets, all_preds = test_run(
        admin=admin, data=data, set_name="test", shuffle=False, save=False
    )

    class_report = {
        "cls_report": cls_report,
        "all_targets": [int(i) for i in all_targets],
        "all_preds": [int(i) for i in all_preds],
    }


    with open(res_out_dir / out1, 'w') as f:
        json.dump(results.model_dump(), f)

    with open(res_out_dir / out2, 'w') as f:
        json.dump(class_report, f)
        
    print(f'Saved to: {out1} and {out2}')
    
    return results, class_report
    
# results, class_report =  run_tests()


#### Otherwise load the results

In [ ]:
with open(res_out_dir / out1, 'r') as f:
    results = json.load(f)

with open(res_out_dir / out2, 'r') as f:
    cls_report = json.load(f)

### Plotting utilities

In [ ]:
from typing import Any


def plot_count_vs_f1_correlation(
    counts: dict[str, int],
    cls_report: dict[str, dict[str, float]],
    class_to_idx: dict[str, int],
    title: str,
    xlabel: str,
    color: str = DEFAULT_ACCENT,
    save_path: str | Path | None = None,
) -> None:
    """Plot correlation between a per-gloss count metric (e.g. instances or unique
    signers) and per-class F1 score.

    Args:
        counts: Dict mapping gloss name -> count for the metric being correlated.
        cls_report: Classification report dict from sklearn (output of test_topk_clsrep).
        class_to_idx: Dict mapping gloss name -> numeric class index.
        title: Plot title.
        xlabel: X-axis label describing the count metric.
        color: Scatter point colour, passed through to `plot_metric_correlation`.
        save_path: Optionally save the figure. Defaults to None.
    """
    matched_counts = []
    f1_scores = []

    for gloss, count in counts.items():
        idx = class_to_idx.get(gloss)
        if idx is None:
            continue
        report_entry = cls_report.get(str(idx))
        if report_entry is None:
            continue
        matched_counts.append(count)
        f1_scores.append(report_entry["f1-score"])

    fig, ax, tau, p = plot_metric_correlation(
        matched_counts,
        f1_scores,
        title=title,
        xlabel=xlabel,
        color=color,
    )
    if save_path is not None:
        save_fig(fig, save_path)
    plt.show()
    print(f"Kendall's tau: {tau:.4f}")
    print(f"p-value: {p:.4e}")


def sort_dict(dict: dict[str, Any]) -> dict[str, Any]:
    return {key: value for key, value in sorted(dict.items(), key=lambda x: x[0])}

def get_num_instances_counts(set_statistics: dict) -> dict[str, int]:
    return {
        gloss : instance['num_instances'] for gloss, instance in set_statistics['per_instance_stats'].items()
    }

def get_signer_counts(set_statistics: dict) -> dict[str, int]:
    """Extract number of unique training signers per gloss from set statistics."""
    return {
        gloss: len(instance['signers_distribution'])
        for gloss, instance in set_statistics['per_instance_stats'].items()
    }


### Correlation of F1-score with instance count:

In [ ]:
clss_to_idx = {gloss: idx for idx, gloss in enumerate(classes)}
plot_count_vs_f1_correlation(
    get_num_instances_counts(per_set_stats['train']),
    cls_report,
    clss_to_idx,
    title="Per-gloss instance count vs. recognition F1 score",
    xlabel="Number of instances",
    save_path=output_dir / f'corr_mat_{model_name}.pdf',
)

### Correlation of F1-score with signer count:

In [ ]:
plot_count_vs_f1_correlation(
    get_signer_counts(per_set_stats['train']),
    cls_report,
    clss_to_idx,
    title="Per-gloss signer count vs. recognition F1 score",
    xlabel="Number of unique signers",
    color=LINE_PALETTE[2],
    save_path=output_dir / f'corr_signers_{model_name}.pdf',
)

## Collect performance outliers

In [ ]:
def get_performance_outliers(
    set_statistics: dict,
    cls_report: dict[str, dict[str, float]],
    class_to_idx: dict[str, int],
    low_signer_thresh: int = 3,
    high_f1_thresh: float = 0.80,
    high_signer_thresh: int = 10,
    low_f1_thresh: float = 0.20
):
    """Extracts and prints overachieving and underachieving glosses."""
    
    data = []
    
    # 1. Compile all data into a single list of dictionaries
    for gloss, instance in set_statistics['per_instance_stats'].items():
        idx = class_to_idx.get(gloss)
        if idx is None:
            continue
            
        report_entry = cls_report.get(str(idx))
        if report_entry is None:
            continue
            
        num_instances = instance['num_instances']
        num_signers = len(instance['signers_distribution'])
        f1_score = report_entry["f1-score"]
        
        data.append({
            "Gloss": gloss,
            "Instances": num_instances,
            "Signers": num_signers,
            "F1_Score": f1_score
        })
        
    df = pd.DataFrame(data)
    
    # 2. Filter for Overachievers (Low data, High F1)
    overachievers = df[
        (df['Signers'] <= low_signer_thresh) & 
        (df['F1_Score'] >= high_f1_thresh)
    ].sort_values(by=['F1_Score', 'Signers'], ascending=[False, True])
    
    # 3. Filter for Underachievers (High data, Low F1)
    underachievers = df[
        (df['Signers'] >= high_signer_thresh) & 
        (df['F1_Score'] <= low_f1_thresh)
    ].sort_values(by=['F1_Score', 'Signers'], ascending=[True, False])
    
    # 4. Print the results nicely
    print("🏆 THE OVERACHIEVERS 🏆")
    print(f"Glosses with <= {low_signer_thresh} signers but F1 >= {high_f1_thresh}")
    print("-" * 50)
    if overachievers.empty:
        print("No glosses met this criteria. (You may need to adjust the thresholds).")
    else:
        print(overachievers.to_string(index=False))
        
    print("\n\n" + "="*50 + "\n")
    
    print("📉 THE UNDERACHIEVERS 📉")
    print(f"Glosses with >= {high_signer_thresh} signers but F1 <= {low_f1_thresh}")
    print("-" * 50)
    if underachievers.empty:
        print("No glosses met this criteria. (You may need to adjust the thresholds).")
    else:
        print(underachievers.to_string(index=False))

    return overachievers, underachievers

# --- How to call the function ---
# Pass the 'train' split stats to see what it learned from!
over_df, under_df = get_performance_outliers(
    set_statistics=per_set_stats['train'], 
    cls_report=cls_report, 
    class_to_idx=clss_to_idx,
    # You can tweak these thresholds if your lists are too long or too short:
    low_signer_thresh=3,   
    high_f1_thresh=0.80,   
    high_signer_thresh=6, 
    low_f1_thresh=0.20     
)

🏆 THE OVERACHIEVERS 🏆
Glosses with <= 3 signers but F1 >= 0.8
--------------------------------------------------
    Gloss  Instances  Signers  F1_Score
beginning          5        3       1.0
 swimsuit          5        3       1.0



📉 THE UNDERACHIEVERS 📉
Glosses with >= 6 signers but F1 <= 0.2
--------------------------------------------------
         Gloss  Instances  Signers  F1_Score
        before         18       13       0.0
          thin         15       13       0.0
           eat         14       13       0.0
        yellow         13       13       0.0
           hot         15       12       0.0
          what         14       12       0.0
          blue         15       12       0.0
       hearing         15       12       0.0
          time         14       12       0.0
         wrong         15       11       0.0
          corn         13       11       0.0
         short         13       11       0.0
          need         13       11       0.0
     different      

### Save performance outliers file

In [ ]:
# 1. Convert the DataFrames into lists of dictionaries
outliers_dict = {
    "overachievers": over_df.to_dict(orient="records"),
    "underachievers": under_df.to_dict(orient="records")
}

# 2. Save the dictionary to a JSON file
fname = "performance_outliers.json"
with open(res_out_dir / fname, "w") as f:
    json.dump(outliers_dict, f, indent=4)

print(f"Successfully saved outliers to {res_out_dir / fname}")

Successfully saved outliers to performance_outliers.json
